# Import Statements

In [33]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bi-lstm-dls/bi_lstm/Bi_LSTM_Model_dls_vocab.json
/kaggle/input/bi-lstm-dls/bi_lstm/bilstm_dls_model.pth
/kaggle/input/processed/train_lemmastop.csv
/kaggle/input/processed/cleaned_train.csv
/kaggle/input/processed/test_lemmastop.csv
/kaggle/input/processed/cleaned_test.csv
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/config.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/training_args.bin
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/tokenizer.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/tokenizer_config.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/model.safetensors
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/special_tokens_map.json
/kaggle/input/d-bert-train/transformers/default/1/d_bert_train/vocab.txt
/kaggle/input/bert-weights/bert-1/config.json
/kaggle/input/bert-weights/bert-1/tokenizer.json
/kaggle/input/bert-weights/bert-1/tokenizer_config.json
/

In [34]:
# !pip install contractions

In [35]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from tqdm.auto import tqdm
import json

# Test Dataset

In [36]:
df_test = pd.read_csv('/kaggle/input/processed/test_lemmastop.csv')
df_test['processed_text'] = df_test['processed_text'].fillna('')
df_test.head()

,id,text,clean_text,processed_text
0,0,she wanted to fight over every single little t...,she wanted to fight over every single little t...,wanted fight every single little thing
1,1,"anyway, back to tuesday.",anyway back to tuesday,anyway back tuesday
2,2,she shrieked at the dog to go back.,she shrieked at the dog to go back,shrieked dog go back
3,3,yelling for everyone to get back or get inside...,yelling for everyone to get back or get inside...,yelling everyone get back get inside draw knif...
4,4,still kind of freaky.,still kind of freaky,still kind freaky


In [37]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']

# Set Device

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Tokenization

In [39]:
def text_to_sequence(text, vocab):
    """Converts a text string to a sequence of integers using the vocab."""
    tokens = word_tokenize(text)
    return [vocab.get(word, vocab.get('<UNK>', 1)) for word in tokens]

In [40]:
def pad_sequence(seq, max_len):
    """Pads a sequence to max_len. Truncates if longer."""
    if len(seq) > max_len:
        return seq[:max_len]  # Truncate
    else:
        return seq + [vocab.get('<PAD>', 0)] * (max_len - len(seq))

# GRU

In [41]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.gru = nn.GRU(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers,     
            bidirectional=False,
            dropout=dropout,
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.embedding(text)
        
        output, hidden = self.gru(embedded)
        final_hidden = hidden[-1, :, :]
        
        final_hidden = self.dropout(final_hidden)
        prediction = self.fc(final_hidden)
        return prediction

In [42]:
# os.environ["WANDB_DISABLED"] = "true"

# Parameters

In [43]:
MAX_LEN = 18
BATCH_SIZE = 32
EMBEDDING_DIM = 100 
HIDDEN_DIM = 128     
OUTPUT_DIM = 5      
N_LAYERS = 2         
DROPOUT = 0.4        
LEARNING_RATE = 1e-3
N_EPOCHS = 50

# Load Vocab and Model

In [45]:
base_path = "/kaggle/input/gru-dls/gru" 

with open(f'{base_path}/gru_dls_vocab.json', 'r') as f:
    vocab = json.load(f)

model = GRUModel(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT)
model.load_state_dict(torch.load(f'{base_path}/gru_dls_model.pth', map_location=device))
model = model.to(device)
model.eval()

GRUModel(
  (embedding): Embedding(4873, 100, padding_idx=0)
  (gru): GRU(100, 128, num_layers=2, batch_first=True, dropout=0.4)
  (fc): Linear(in_features=128, out_features=5, bias=True)
  (dropout): Dropout(p=0.4, inplace=False)
)

In [46]:
sequences = [text_to_sequence(text, vocab) for text in df_test['processed_text']]
test_sequences = [pad_sequence(s, MAX_LEN) for s in sequences]
X_test = torch.tensor(test_sequences, dtype=torch.long)

test_data = torch.utils.data.TensorDataset(X_test)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

# Make Predictions

In [47]:
all_preds = []

print("Starting predictions.")
with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()
        
        all_preds.extend(preds.cpu().numpy())

Starting predictions.


# Submission

In [48]:
df_submission = pd.DataFrame(all_preds, columns=emotion_cols)
df_submission.insert(0, 'id', df_test['id'])
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,1,0,0,0
1,1,0,0,0,0,0
2,2,1,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1


In [49]:
df_submission.to_csv('submission.csv', index=False)